# CUDA Nonce Finder - Google Colab

This notebook allows you to run the CUDA nonce finder on Google Colab's free GPU.

**Before running:**
1. Go to Runtime → Change runtime type
2. Select GPU (T4 recommended)
3. Save

## Step 1: Check GPU Availability

In [ ]:
!nvidia-smi

## Step 2: Create SHA1 Implementation Files

The SHA1 implementation is from [mochimodev/cuda-hashing-algos](https://github.com/mochimodev/cuda-hashing-algos) (Public Domain).

We create 3 files with the exact original structure:
1. `config.h` - Type definitions
2. `sha1.cuh` - Header with function declaration
3. `sha1.cu` - Implementation

In [ ]:
%%writefile config.h
/*
 * Type Definitions for CUDA Hashing Algos
 *
 * Date: 12 June 2019
 * Revision: 1
 *
 * This file is released into the Public Domain.
 */

#pragma once
#define USE_MD2 1
#define USE_MD5 1
#define USE_SHA1 1
#define USE_SHA256 1

#define CUDA_HASH 1
#define OCL_HASH 0

typedef unsigned char BYTE;
typedef unsigned int  WORD;
typedef unsigned long long LONG;

#include <stdlib.h>
#include <string.h>
#include <stdio.h>

In [ ]:
%%writefile sha1.cuh
/*
 * sha1.cuh CUDA Implementation of SHA1 Hashing    
 *
 * Date: 12 June 2019
 * Revision: 1
 * 
 * Based on the public domain Reference Implementation in C, by
 * Brad Conte, original code here:
 *
 * https://github.com/B-Con/crypto-algorithms
 *
 * This file is released into the Public Domain.
 */

 
#pragma once
#include "config.h"
void mcm_cuda_sha1_hash_batch(BYTE* in, WORD inlen, BYTE* out, WORD n_batch);

In [ ]:
%%writefile sha1.cu
/*
 * sha1.cu CUDA Implementation of SHA1 Hashing
 *
 * Date: 12 June 2019
 * Revision: 1
 *
 * Based on the public domain Reference Implementation in C, by
 * Brad Conte, original code here:
 *
 * https://github.com/B-Con/crypto-algorithms
 *
 * This file is released into the Public Domain.
 */


/*************************** HEADER FILES ***************************/
#include <stdlib.h>
#include <memory.h>
extern "C" {
#include "sha1.cuh"
}

/****************************** MACROS ******************************/
#define SHA1_BLOCK_SIZE 20              // SHA1 outputs a 20 byte digest

/**************************** DATA TYPES ****************************/
typedef struct {
	BYTE data[64];
	WORD datalen;
	unsigned long long bitlen;
	WORD state[5];
	WORD k[4];
} CUDA_SHA1_CTX;

/****************************** MACROS ******************************/
#ifndef ROTLEFT
#define ROTLEFT(a,b) (((a) << (b)) | ((a) >> (32-(b))))
#endif

/*********************** FUNCTION DEFINITIONS ***********************/
__device__  __forceinline__ void cuda_sha1_transform(CUDA_SHA1_CTX *ctx, const BYTE data[])
{
	WORD a, b, c, d, e, i, j, t, m[80];

	for (i = 0, j = 0; i < 16; ++i, j += 4)
		m[i] = (data[j] << 24) + (data[j + 1] << 16) + (data[j + 2] << 8) + (data[j + 3]);
	for ( ; i < 80; ++i) {
		m[i] = (m[i - 3] ^ m[i - 8] ^ m[i - 14] ^ m[i - 16]);
		m[i] = (m[i] << 1) | (m[i] >> 31);
	}

	a = ctx->state[0];
	b = ctx->state[1];
	c = ctx->state[2];
	d = ctx->state[3];
	e = ctx->state[4];

	for (i = 0; i < 20; ++i) {
		t = ROTLEFT(a, 5) + ((b & c) ^ (~b & d)) + e + ctx->k[0] + m[i];
		e = d;
		d = c;
		c = ROTLEFT(b, 30);
		b = a;
		a = t;
	}
	for ( ; i < 40; ++i) {
		t = ROTLEFT(a, 5) + (b ^ c ^ d) + e + ctx->k[1] + m[i];
		e = d;
		d = c;
		c = ROTLEFT(b, 30);
		b = a;
		a = t;
	}
	for ( ; i < 60; ++i) {
		t = ROTLEFT(a, 5) + ((b & c) ^ (b & d) ^ (c & d))  + e + ctx->k[2] + m[i];
		e = d;
		d = c;
		c = ROTLEFT(b, 30);
		b = a;
		a = t;
	}
	for ( ; i < 80; ++i) {
		t = ROTLEFT(a, 5) + (b ^ c ^ d) + e + ctx->k[3] + m[i];
		e = d;
		d = c;
		c = ROTLEFT(b, 30);
		b = a;
		a = t;
	}

	ctx->state[0] += a;
	ctx->state[1] += b;
	ctx->state[2] += c;
	ctx->state[3] += d;
	ctx->state[4] += e;
}

__device__ void cuda_sha1_init(CUDA_SHA1_CTX *ctx)
{
	ctx->datalen = 0;
	ctx->bitlen = 0;
	ctx->state[0] = 0x67452301;
	ctx->state[1] = 0xEFCDAB89;
	ctx->state[2] = 0x98BADCFE;
	ctx->state[3] = 0x10325476;
	ctx->state[4] = 0xc3d2e1f0;
	ctx->k[0] = 0x5a827999;
	ctx->k[1] = 0x6ed9eba1;
	ctx->k[2] = 0x8f1bbcdc;
	ctx->k[3] = 0xca62c1d6;
}

__device__ void cuda_sha1_update(CUDA_SHA1_CTX *ctx, const BYTE data[], size_t len)
{
	size_t i;

	for (i = 0; i < len; ++i) {
		ctx->data[ctx->datalen] = data[i];
		ctx->datalen++;
		if (ctx->datalen == 64) {
			cuda_sha1_transform(ctx, ctx->data);
			ctx->bitlen += 512;
			ctx->datalen = 0;
		}
	}
}

__device__ void cuda_sha1_final(CUDA_SHA1_CTX *ctx, BYTE hash[])
{
	WORD i;

	i = ctx->datalen;

	// Pad whatever data is left in the buffer.
	if (ctx->datalen < 56) {
		ctx->data[i++] = 0x80;
		while (i < 56)
			ctx->data[i++] = 0x00;
	}
	else {
		ctx->data[i++] = 0x80;
		while (i < 64)
			ctx->data[i++] = 0x00;
		cuda_sha1_transform(ctx, ctx->data);
		memset(ctx->data, 0, 56);
	}

	// Append to the padding the total message's length in bits and transform.
	ctx->bitlen += ctx->datalen * 8;
	ctx->data[63] = ctx->bitlen;
	ctx->data[62] = ctx->bitlen >> 8;
	ctx->data[61] = ctx->bitlen >> 16;
	ctx->data[60] = ctx->bitlen >> 24;
	ctx->data[59] = ctx->bitlen >> 32;
	ctx->data[58] = ctx->bitlen >> 40;
	ctx->data[57] = ctx->bitlen >> 48;
	ctx->data[56] = ctx->bitlen >> 56;
	cuda_sha1_transform(ctx, ctx->data);

	// Since this implementation uses little endian byte ordering and MD uses big endian,
	// reverse all the bytes when copying the final state to the output hash.
	for (i = 0; i < 4; ++i) {
		hash[i]      = (ctx->state[0] >> (24 - i * 8)) & 0x000000ff;
		hash[i + 4]  = (ctx->state[1] >> (24 - i * 8)) & 0x000000ff;
		hash[i + 8]  = (ctx->state[2] >> (24 - i * 8)) & 0x000000ff;
		hash[i + 12] = (ctx->state[3] >> (24 - i * 8)) & 0x000000ff;
		hash[i + 16] = (ctx->state[4] >> (24 - i * 8)) & 0x000000ff;
	}
}

__global__ void kernel_sha1_hash(BYTE* indata, WORD inlen, BYTE* outdata, WORD n_batch)
{
	WORD thread = blockIdx.x * blockDim.x + threadIdx.x;
	if (thread >= n_batch)
	{
		return;
	}
	BYTE* in = indata  + thread * inlen;
	BYTE* out = outdata  + thread * SHA1_BLOCK_SIZE;
	CUDA_SHA1_CTX ctx;
	cuda_sha1_init(&ctx);
	cuda_sha1_update(&ctx, in, inlen);
	cuda_sha1_final(&ctx, out);
}

extern "C"
{
void mcm_cuda_sha1_hash_batch(BYTE* in, WORD inlen, BYTE* out, WORD n_batch)
{
	BYTE *cuda_indata;
	BYTE *cuda_outdata;
	cudaMalloc(&cuda_indata, inlen * n_batch);
	cudaMalloc(&cuda_outdata, SHA1_BLOCK_SIZE * n_batch);
	cudaMemcpy(cuda_indata, in, inlen * n_batch, cudaMemcpyHostToDevice);

	WORD thread = 256;
	WORD block = (n_batch + thread - 1) / thread;

	kernel_sha1_hash << < block, thread >> > (cuda_indata, inlen, cuda_outdata, n_batch);
	cudaMemcpy(out, cuda_outdata, SHA1_BLOCK_SIZE * n_batch, cudaMemcpyDeviceToHost);
	cudaDeviceSynchronize();
	cudaError_t error = cudaGetLastError();
	if (error != cudaSuccess) {
		printf("Error cuda sha1 hash: %s \n", cudaGetErrorString(error));
	}
	cudaFree(cuda_indata);
	cudaFree(cuda_outdata);
}
}

In [ ]:
%%writefile nonce_finder.cu
// CUDA Nonce Finder - finds nonce where SHA1(data + nonce) ends with target suffix
// SHA1 from: https://github.com/mochimodev/cuda-hashing-algos (Public Domain)

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <cuda_runtime.h>

#include "sha1.cu"

#define MAX_DATA_SIZE 256
#define MAX_SUFFIX_SIZE 8

__constant__ uint8_t const_data[MAX_DATA_SIZE];
__constant__ int const_data_len;
__constant__ uint8_t const_suffix[MAX_SUFFIX_SIZE];
__constant__ int const_suffix_len;

__device__ bool hash_ends_with_suffix(const BYTE* hash) {
    for (int i = 0; i < const_suffix_len; i++) {
        if (hash[SHA1_BLOCK_SIZE - const_suffix_len + i] != const_suffix[i])
            return false;
    }
    return true;
}

__global__ void find_nonce_constant_memory(uint64_t start_nonce, int nonces_per_thread,
                                           int* found_flag, uint64_t* found_nonce) {
    uint64_t thread_id = blockIdx.x * blockDim.x + threadIdx.x;
    uint64_t total_threads = gridDim.x * blockDim.x;

    for (int i = 0; i < nonces_per_thread; i++) {
        if (*found_flag) return;

        uint64_t nonce = start_nonce + thread_id + i * total_threads;

        BYTE message[MAX_DATA_SIZE + 8];
        for (int j = 0; j < const_data_len; j++)
            message[j] = const_data[j];

        for (int j = 0; j < 8; j++)
            message[const_data_len + j] = (nonce >> (j * 8)) & 0xFF;

        BYTE hash[SHA1_BLOCK_SIZE];
        CUDA_SHA1_CTX ctx;
        cuda_sha1_init(&ctx);
        cuda_sha1_update(&ctx, message, const_data_len + 8);
        cuda_sha1_final(&ctx, hash);

        if (hash_ends_with_suffix(hash)) {
            if (atomicExch(found_flag, 1) == 0)
                *found_nonce = nonce;
            return;
        }
    }
}

__global__ void find_nonce_shared_memory(uint64_t start_nonce, int nonces_per_thread,
                                         int* found_flag, uint64_t* found_nonce) {
    __shared__ BYTE shared_data[MAX_DATA_SIZE];
    __shared__ int shared_data_len;

    if (threadIdx.x == 0)
        shared_data_len = const_data_len;

    for (int i = threadIdx.x; i < const_data_len; i += blockDim.x)
        shared_data[i] = const_data[i];

    __syncthreads();

    uint64_t thread_id = blockIdx.x * blockDim.x + threadIdx.x;
    uint64_t total_threads = gridDim.x * blockDim.x;

    for (int i = 0; i < nonces_per_thread; i++) {
        if (*found_flag) return;

        uint64_t nonce = start_nonce + thread_id + i * total_threads;

        BYTE message[MAX_DATA_SIZE + 8];
        for (int j = 0; j < shared_data_len; j++)
            message[j] = shared_data[j];

        for (int j = 0; j < 8; j++)
            message[shared_data_len + j] = (nonce >> (j * 8)) & 0xFF;

        BYTE hash[SHA1_BLOCK_SIZE];
        CUDA_SHA1_CTX ctx;
        cuda_sha1_init(&ctx);
        cuda_sha1_update(&ctx, message, shared_data_len + 8);
        cuda_sha1_final(&ctx, hash);

        if (hash_ends_with_suffix(hash)) {
            if (atomicExch(found_flag, 1) == 0)
                *found_nonce = nonce;
            return;
        }
    }
}

#define CUDA_CHECK(call) { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        printf("CUDA error: %s\n", cudaGetErrorString(err)); \
        exit(1); \
    } \
}

int main(int argc, char** argv) {
    uint8_t input_data[MAX_DATA_SIZE] = "HELLO";
    int input_data_len = 5;
    uint8_t target_suffix[MAX_SUFFIX_SIZE] = {0xFF};
    int target_suffix_len = 1;
    int num_blocks = 1024;
    int threads_per_block = 256;
    int nonces_per_thread = 100;
    bool use_shared_memory = false;

    for (int i = 1; i < argc; i++) {
        if (!strcmp(argv[i], "--data") && i+1 < argc) {
            char* str = argv[++i];
            input_data_len = strlen(str);
            for (int j = 0; j < input_data_len; j++)
                input_data[j] = str[j];
        }
        else if (!strcmp(argv[i], "--suffix") && i+1 < argc) {
            char* str = argv[++i];
            target_suffix_len = strlen(str);
            for (int j = 0; j < target_suffix_len; j++)
                target_suffix[j] = str[j];
        }
        else if (!strcmp(argv[i], "--blocks") && i+1 < argc)
            num_blocks = atoi(argv[++i]);
        else if (!strcmp(argv[i], "--threads") && i+1 < argc)
            threads_per_block = atoi(argv[++i]);
        else if (!strcmp(argv[i], "--per-thread") && i+1 < argc)
            nonces_per_thread = atoi(argv[++i]);
        else if (!strcmp(argv[i], "--shared"))
            use_shared_memory = true;
    }

    cudaDeviceProp gpu_properties;
    cudaGetDeviceProperties(&gpu_properties, 0);
    printf("GPU: %s\n", gpu_properties.name);
    printf("Suffix length: %d bytes\n", target_suffix_len);
    printf("Using %s memory\n\n", use_shared_memory ? "shared" : "constant");

    CUDA_CHECK(cudaMemcpyToSymbol(const_data, input_data, input_data_len));
    CUDA_CHECK(cudaMemcpyToSymbol(const_data_len, &input_data_len, sizeof(int)));
    CUDA_CHECK(cudaMemcpyToSymbol(const_suffix, target_suffix, target_suffix_len));
    CUDA_CHECK(cudaMemcpyToSymbol(const_suffix_len, &target_suffix_len, sizeof(int)));

    int* gpu_found_flag;
    uint64_t* gpu_found_nonce;
    CUDA_CHECK(cudaMalloc(&gpu_found_flag, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&gpu_found_nonce, sizeof(uint64_t)));
    CUDA_CHECK(cudaMemset(gpu_found_flag, 0, sizeof(int)));

    uint64_t current_nonce = 0;
    uint64_t nonces_per_kernel = (uint64_t)num_blocks * threads_per_block * nonces_per_thread;
    uint64_t total_hashes = 0;
    int found = 0;

    cudaEvent_t timer_start, timer_end;
    cudaEventCreate(&timer_start);
    cudaEventCreate(&timer_end);
    cudaEventRecord(timer_start);

    printf("Searching...\n");
    while (!found) {
        if (current_nonce > UINT64_MAX - nonces_per_kernel) {
            printf("Error: Exhausted all possible nonces.\n");
            break;
        }

        if (use_shared_memory)
            find_nonce_shared_memory<<<num_blocks, threads_per_block>>>(
                current_nonce, nonces_per_thread, gpu_found_flag, gpu_found_nonce);
        else
            find_nonce_constant_memory<<<num_blocks, threads_per_block>>>(
                current_nonce, nonces_per_thread, gpu_found_flag, gpu_found_nonce);

        CUDA_CHECK(cudaDeviceSynchronize());
        CUDA_CHECK(cudaMemcpy(&found, gpu_found_flag, sizeof(int), cudaMemcpyDeviceToHost));

        total_hashes += nonces_per_kernel;
        current_nonce += nonces_per_kernel;

        if (total_hashes % 100000000 < nonces_per_kernel)
            printf("  %lu M hashes...\n", total_hashes / 1000000);
    }

    cudaEventRecord(timer_end);
    cudaEventSynchronize(timer_end);
    float elapsed_ms;
    cudaEventElapsedTime(&elapsed_ms, timer_start, timer_end);

    if (found) {
        uint64_t result_nonce;
        CUDA_CHECK(cudaMemcpy(&result_nonce, gpu_found_nonce, sizeof(uint64_t), cudaMemcpyDeviceToHost));

        printf("\nFound nonce: %lu\n", result_nonce);
        printf("Total hashes: %lu\n", total_hashes);
        printf("Time: %.2f sec\n", elapsed_ms / 1000);
        printf("Rate: %.2f MH/s\n", (total_hashes / 1e6) / (elapsed_ms / 1000));
    }

    cudaFree(gpu_found_flag);
    cudaFree(gpu_found_nonce);

    return 0;
}

## Step 4: Compile the CUDA Program

In [ ]:
!nvcc -O3 -use_fast_math -arch=sm_75 -o nonce_finder nonce_finder.cu 2>&1

## Step 5: Run the Nonce Finder

Find a nonce where SHA1("HELLO" + nonce) ends in a specific suffix.

Note: Use short suffixes (1-2 bytes) for quick results. Longer suffixes take exponentially longer!

In [ ]:
# Find a nonce where SHA1("HELLO" + nonce) ends with "A"
!./nonce_finder --data "HELLO" --suffix "A"

## Step 6: Verify the Result

Use Python to verify the found nonce

In [ ]:
import hashlib
import struct

def verify_nonce(data_string, nonce, suffix_string):
    data = data_string.encode('utf-8')
    suffix = suffix_string.encode('utf-8')
    nonce_bytes = struct.pack('<Q', nonce)  # 8 bytes, little-endian
    
    message = data + nonce_bytes
    hash_result = hashlib.sha1(message).digest()

    print(f"DATA:        \"{data_string}\"")
    print(f"NONCE:       {nonce}")
    print(f"SHA1:        {hash_result.hex()}")
    print(f"SUFFIX:      \"{suffix_string}\"")
    print(f"Hash ends:   \"{hash_result[-len(suffix):].decode('latin-1')}\"")
    print()

    if hash_result[-len(suffix):] == suffix:
        print("SUCCESS!")
    else:
        print("FAILURE!")

# Example: verify_nonce("HELLO", 123, "A")
verify_nonce("HELLO", 47, "A")

## Experiment: Try Different Configurations

In [ ]:
# Try with different configurations
print("=== Constant memory (default) ===")
!./nonce_finder --data "HELLO" --suffix "AB"

print("\n=== Shared memory ===")
!./nonce_finder --data "HELLO" --suffix "AB" --shared

print("\n=== More blocks and threads ===")
!./nonce_finder --data "HELLO" --suffix "AB" --blocks 2048 --threads 512

## Challenge: Find a 2-byte suffix (harder!)

This requires ~65,536 times more work!

In [ ]:
# This will take longer - searching for a 2-character suffix
!./nonce_finder --data "Hello guys" --suffix "XY"